## **The Degradation Problem in Deep Networks**

**Observation**

Intuitively, a deeper network should be at least as good as a shallower one: the extra layers could, in principle, just learn to pass their input through unchanged. Yet in practice, when you stack many layers in a **plain** (non-residual) network, accuracy first saturates and then **degrades** — and crucially, this happens on the **training set**, not just the test set.

He et al. (2015) showed that a 56-layer plain CNN has **higher training error** than a 20-layer one on CIFAR-10. More depth made the model strictly worse at fitting the data it was trained on.

### **This is NOT overfitting**

It is tempting to blame overfitting, but the evidence rules it out:

* Overfitting means **low** training error and **high** test error. Here the **training** error itself is higher for the deeper net.
* It is also not (primarily) vanishing/exploding gradients — these networks already used Batch Normalization and proper initialization, which keep gradients healthy.

The degradation problem is an **optimization difficulty**: the deeper plain network is harder to optimize, and SGD struggles to drive its training error down to the level the shallower network reaches.

### **The identity-mapping argument**

Consider a shallow network that already achieves some training error. Build a deeper network by copying it and adding extra layers on top. If those extra layers could learn the **identity mapping** ($f(x) = x$), the deeper network would match the shallower one exactly — so it should never be *worse*.

The fact that plain deep nets *are* worse means **the optimizer cannot easily find the identity mapping** through stacks of nonlinear layers. Learning $H(x) = x$ from scratch with several conv + ReLU layers turns out to be hard.

### **Residual connections as the fix**

Residual learning makes the identity mapping the **default**. Instead of asking a block to learn the full target $H(x)$, we ask it to learn the **residual** $F(x) = H(x) - x$ and add the input back via a skip connection:

$$
y = F(x, \{W_i\}) + x
$$

Now if the optimal function is the identity, the block only needs to push $F(x) \to 0$ (drive the weights toward zero) — far easier than fitting an identity through nonlinearities. Extra depth can no longer hurt: a layer that has nothing useful to add can simply learn $F \approx 0$.

```python
import torch.nn as nn

class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(channels)
        self.relu  = nn.ReLU(inplace=True)

    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = out + identity          # skip connection: learn the residual
        return self.relu(out)
```

With this change, He et al. successfully trained networks of 50, 101, even 152 layers — and deeper became *better*, reversing the degradation.

### **Takeaways and related topics**

* The **degradation problem** is an optimization phenomenon: very deep *plain* networks get **worse training accuracy**, not because of overfitting or (mainly) vanishing gradients, but because the optimizer cannot easily learn identity-like mappings.
* **Residual / skip connections** solve this by reparameterizing layers to learn a residual, making identity the easy default and letting gradients flow directly through the skip path.
* See also: the **residual_connections** topic for the mechanism in detail, and **cnn_architectures/resnet** for the full ResNet architecture built from these blocks.

**Reference:** He, Zhang, Ren, Sun, *Deep Residual Learning for Image Recognition* (ResNet), 2015.